In [1]:
# Brian Pitcher
# CS-340

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from aac_crud import AnimalShelterCRUD

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "SNHU1234"
HOST = 'nv-desktop-services.apporto.com'
PORT = 31230
DB = 'AAC'
COL = 'animals'

# Connect to database via CRUD Module
db = AnimalShelterCRUD(username, password, HOST, PORT, DB, COL)



import pandas as pd

# Load your CSV file directly
df = pd.read_csv('/home/brianpitcher_snhu/CS340_Project/Project_Two/aac_shelter_outcomes.csv')

# Check if '_id' exists — remove it if it does
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)




## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso_Salvare_Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div(className='row',
            style={'display' : 'flex', 'margin' : '10px'},
                 children=[
                     html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),style={'height':'5%', 'width':'5%'}),
                     html.Center(html.B(html.H1('Grazioso Salvare Dashboard - CS340 - Brian Pitcher')))
                               
            ]),
    html.Hr(),
    html.Div(
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.

     dcc.RadioItems(
         id='filter-type',
         options=[
             {'label': '  Water Rescue  ', 'value': 'Water'},
             {'label': '  Mountain and Wilderness Rescue  ', 'value': 'Mountain'},
             {'label': '  Disaster and Individual Rescue  ', 'value': 'Disaster'},
             {'label': '  Reset', 'value': 'Reset'},      
         ],
         value='Reset'
    )     
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         column_selectable="single",
                         row_selectable="single",
                         row_deletable=False,
                         selected_columns=[],
                         selected_rows=[0],
                         page_action="native",
                         page_current= 0,
                         page_size= 10,
                         ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type is None or filter_type == 'Reset':
        return df[df['animal_type'] == 'Dog'].to_dict('records')

    filtered_df = df[df['animal_type'] == 'Dog']  # Start with only dogs

    if filter_type == 'Water':
        filtered_df = filtered_df[
            (filtered_df['breed'].str.contains('Labrador Retriever Mix|Chesapeake|Newfoundland', na=False)) &
            (filtered_df['sex_upon_outcome'] == 'Intact Female') &
            (filtered_df['age_upon_outcome_in_weeks'] > 26) &
            (filtered_df['age_upon_outcome_in_weeks'] < 156)
        ]

    elif filter_type == 'Mountain':
        filtered_df = filtered_df[
            (filtered_df['breed'].str.contains('German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler', na=False)) &
            (filtered_df['sex_upon_outcome'] == 'Intact Male') &
            (filtered_df['age_upon_outcome_in_weeks'] > 26) &
            (filtered_df['age_upon_outcome_in_weeks'] < 156)
        ]

    elif filter_type == 'Disaster':
        filtered_df = filtered_df[
            (filtered_df['breed'].str.contains('Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler', na=False)) &
            (filtered_df['sex_upon_outcome'] == 'Intact Male') &
            (filtered_df['age_upon_outcome_in_weeks'] > 20) &
            (filtered_df['age_upon_outcome_in_weeks'] < 300)
        ]

    return filtered_df.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None:
        return
    
    df = pd.DataFrame.from_records(viewData)
    return [
        dcc.Graph(
            figure = px.pie(df, names='breed', title='Preferred Animals')
        )
    ]
 
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run_server(debug=True)


Dash app running on http://127.0.0.1:28940/
